# Plot the top concepts contributing for the positive classification for a give attribute.
requirements:
- CelebA dataset
- SAE activations as a .pth file
- concept names for RN-50
- trained final model for relevant attribute as .pt files

#### Provide paths

In [51]:
from pathlib import Path

CELEBA_ROOT      = "./data/celeba"                          # CelebA root
SAE_ACT_TRAIN    = "./data/activations_img/celeba/clip_RN50/out/train/sae_activations.pth"
CONCEPT_NAMES_CSV = "./checkpoints/clip_RN50_concept_name.csv"

PROBE_CKPT = {
    "Blond_Hair": "./train_linear_probe_celeba/outputs/Blond_Hair/binary_probe.pt",
    "Male":       "./train_linear_probe_celeba/outputs/Male/binary_probe.pt",
    "Eyeglasses": "./train_linear_probe_celeba/outputs/Eyeglasses/binary_probe.pt",
    "Pale_Skin": "./train_linear_probe_celeba/outputs/Pale_Skin/binary_probe.pt",
    # "Attractive": "./train_linear_probe_celeba/outputs/Attractive/binary_probe.pt", # Pre-unlearning concepts
    "Attractive": "unlearning/outputs/Attractive/binary_probe_unlearned_k100_polpositive.pt", # Posy-unlearning concepts
    "Smiling": "./train_linear_probe_celeba/outputs/Smiling/binary_probe.pt",
    "Mouth_Slightly_Open": "./train_linear_probe_celeba/outputs/Mouth_Slightly_Open/binary_probe.pt"
}

ATTRIBUTE     = "Attractive"   # ← switch to "Male" to see that attribute
N_TOP_CONCEPTS = 3             # top concepts per class to show
N_IMAGES       = 4             # top activating images per concept
SPLIT          = "train"       # which split's activations to use

#### Imports & helpers

In [54]:
import csv, os
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


# ── load concept name lookup ──────────────────────────────────────
def load_concept_names(csv_path):
    names = {}
    with open(csv_path, newline="") as f:
        for row in csv.reader(f):
            if len(row) >= 2:
                try:
                    names[int(row[0])] = row[1].strip()
                except ValueError:
                    pass
    return names

# ── load CelebA image file list for a split ───────────────────────
def load_celeba_filenames(celeba_root, split="train"):
    split_map = {"train": 0, "val": 1, "test": 2}
    partition_idx = split_map[split]
    split_file = Path(celeba_root) / "list_eval_partition.txt"
    img_files = []
    with open(split_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2 and int(parts[1]) == partition_idx:
                img_files.append(parts[0])
    return img_files   # list of filenames, index-aligned with sae_activations

# ── load CelebA attribute labels for a split ─────────────────────
def load_celeba_labels(celeba_root, split="train", attribute="Blond_Hair"):
    attr_file = Path(celeba_root) / "list_attr_celeba.txt"
    with open(attr_file) as f:
        lines = f.readlines()
    attr_names = lines[1].strip().split()
    attr_idx   = attr_names.index(attribute)
    partition_file = Path(celeba_root) / "list_eval_partition.txt"
    split_map = {"train": 0, "val": 1, "test": 2}
    target = split_map[split]
    with open(partition_file) as f:
        partitions = [int(l.strip().split()[-1]) for l in f if l.strip()]
    labels = []
    for img_idx, part in enumerate(partitions):
        if part == target:
            row = lines[2 + img_idx].strip().split()
            val = int(row[attr_idx + 1])   # ← +1 to skip filename
            labels.append(1 if val == 1 else 0)
    return torch.tensor(labels, dtype=torch.long)

# ── load probe weights ────────────────────────────────────────────
def load_probe_weights(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "weights" in ckpt:
        w = ckpt["weights"]
    elif "weights_after" in ckpt:
        w = ckpt["weights_after"]
    elif "model_state" in ckpt:
        w = ckpt["model_state"]["linear.weight"]
    else:
        raise KeyError(f"Cannot find weights in checkpoint. Keys: {list(ckpt.keys())}")                # shape [n_concepts]
    return w.squeeze()

#### Load concept names, data, model weights

In [ ]:
concept_names = load_concept_names(CONCEPT_NAMES_CSV)

# SAE activations: [N_images, 8192]
sae_acts = torch.load(SAE_ACT_TRAIN, map_location="cpu")
if sae_acts.ndim == 3 and sae_acts.shape[1] == 1:
    sae_acts = sae_acts.squeeze(1)
print(f"SAE activations: {sae_acts.shape}")

# Image filenames and labels
img_files = load_celeba_filenames(CELEBA_ROOT, split=SPLIT)
labels    = load_celeba_labels(CELEBA_ROOT, split=SPLIT, attribute=ATTRIBUTE)
assert sae_acts.shape[0] == len(labels), \
    f"Mismatch: {sae_acts.shape[0]} activations vs {len(labels)} labels"

# Probe weights
weights = load_probe_weights(PROBE_CKPT[ATTRIBUTE])   # [8192]
print(f"Probe weights: {weights.shape}")

# Split indices by label
pos_idx = (labels == 1).nonzero(as_tuple=True)[0]   # positive class (e.g. blonde)
neg_idx = (labels == 0).nonzero(as_tuple=True)[0]   # negative class

pos_label = ATTRIBUTE.replace("_", " ")     # e.g. "Blond Hair"
neg_label = f"Not {pos_label}"
print(f"Positive ({pos_label}): {len(pos_idx)}  |  Negative ({neg_label}): {len(neg_idx)}")

#### Find top concepts for each class

In [56]:
# Positive concepts  → highest positive weights (predicts positive class)
# Negative concepts  → most negative weights   (predicts negative class)

sorted_pos_concepts = torch.argsort(weights, descending=True)[:N_TOP_CONCEPTS]   # positive weights
sorted_neg_concepts = torch.argsort(weights, descending=False)[:N_TOP_CONCEPTS]  # most negative weights

def get_top_images(concept_idx, image_indices, n=8):
    """Return indices (into the full split) of the N images with highest concept activation,
    restricted to image_indices (either pos_idx or neg_idx)."""
    acts = sae_acts[image_indices, concept_idx]   # activation of this concept for subset
    top_local = torch.argsort(acts, descending=True)[:n]
    return image_indices[top_local]               # global indices

def load_image(filename):
    path = Path(CELEBA_ROOT) / "img_align_celeba" / filename
    return Image.open(path).convert("RGB")

print("Top concepts for", pos_label)
for c in sorted_pos_concepts:
    print(f"  Concept {int(c):5d} | {concept_names.get(int(c), '?'):25s} | weight {weights[c]:+.4f}")

print(f"\nTop concepts for {neg_label}")
for c in sorted_neg_concepts:
    print(f"  Concept {int(c):5d} | {concept_names.get(int(c), '?'):25s} | weight {weights[c]:+.4f}")

Top concepts for Attractive
  Concept  3129 | lace                      | weight +10.5760
  Concept  2761 | golfing                   | weight +10.5237
  Concept  7356 | inform                    | weight +10.4022

Top concepts for Not Attractive
  Concept  6294 | firefighters              | weight -12.2439
  Concept  5371 | officers                  | weight -11.8318
  Concept  2740 | ecard                     | weight -10.3060


In [57]:
# # Sanity check: Compare original vs unlearned weights side by side
# import torch
#
# ORIGINAL_CKPT  = "./train_linear_probe_celeba/outputs/Mouth_Slightly_Open/binary_probe.pt"
# UNLEARNED_CKPT = "./unlearning/outputs/Mouth_Slightly_Open/binary_probe_unlearned_k100.pt"
#
# # Load original weights
# orig_ckpt    = torch.load(ORIGINAL_CKPT, map_location="cpu")
# orig_weights = orig_ckpt["weights"].squeeze()             # [n_concepts]
#
# # Load unlearned weights
# unl_ckpt    = torch.load(UNLEARNED_CKPT, map_location="cpu")
# unl_weights = unl_ckpt["model_state"]["linear.weight"].squeeze()  # [n_concepts]
#
# TOP_CONCEPT = 7406   # from all_concepts_ranked.csv
#
# print(f"{'Concept':>8} | {'Original w':>12} | {'Unlearned w':>12} | {'Δ':>12} | {'Δ%':>8}")
# print("-" * 60)
#
# # Show top-3 positive + top-3 negative concepts by original weight
# for c in list(sorted_pos_concepts) + list(sorted_neg_concepts):
#     i = int(c)
#     o = orig_weights[i].item()
#     u = unl_weights[i].item()
#     d = u - o
#     pct = 100 * d / (abs(o) + 1e-9)
#     name = concept_names.get(i, f"c{i}")
#     print(f"  {i:>6} ({name:<20}) | {o:>+12.4f} | {u:>+12.4f} | {d:>+12.4f} | {pct:>+7.1f}%")
#
# # Always show top concept
# print()
# i = TOP_CONCEPT
# o = orig_weights[i].item()
# u = unl_weights[i].item()
# d = u - o
# pct = 100 * d / (abs(o) + 1e-9)
# print(f"TOP CONCEPT {i} ({concept_names.get(i,'?')}):")
# print(f"  Original  : {o:+.4f}")
# print(f"  Unlearned : {u:+.4f}")
# print(f"  Δ         : {d:+.4f}  ({pct:+.1f}%)")

 Concept |   Original w |  Unlearned w |            Δ |       Δ%
------------------------------------------------------------
    3129 (lace                ) |      -8.7263 |      -5.0965 |      +3.6299 |   +41.6%
    2761 (golfing             ) |      +1.6703 |      +0.4067 |      -1.2636 |   -75.7%
    7356 (inform              ) |      +0.8837 |      +1.9391 |      +1.0554 |  +119.4%
    6294 (firefighters        ) |      -0.0031 |      -0.3675 |      -0.3644 | -11838.8%
    5371 (officers            ) |      -0.7266 |      -5.1102 |      -4.3836 |  -603.3%
    2740 (ecard               ) |      -0.4221 |      +0.0465 |      +0.4685 |  +111.0%

TOP CONCEPT 7406 (smiling):
  Original  : +23.7415
  Unlearned : +18.6942
  Δ         : -5.0473  (-21.3%)


#### Build the grid

In [58]:
img_files_arr = np.array(img_files)

def concept_row_images(concept_idx, class_indices, n=8):
    top_idx = get_top_images(concept_idx, class_indices, n=n)
    images  = [load_image(img_files_arr[i]) for i in top_idx]
    return images, top_idx.tolist()   # ← also return indices

# Collect all images
pos_images_per_concept = [concept_row_images(c, pos_idx, N_IMAGES) for c in sorted_pos_concepts]
neg_images_per_concept = [concept_row_images(c, neg_idx, N_IMAGES) for c in sorted_neg_concepts]

# ── plot ──────────────────────────────────────────────────────────
IMG_SIZE  = 1.5        # inches per image cell
PAD       = 0.05
LABEL_W   = 2.2        # inches for the concept-name label column
GROUP_GAP = 0.6        # gap between the two class groups

total_w = LABEL_W + N_IMAGES * IMG_SIZE + GROUP_GAP + N_IMAGES * IMG_SIZE
total_h = N_TOP_CONCEPTS * IMG_SIZE + 0.6   # + header row

fig = plt.figure(figsize=(total_w, total_h))

# One GridSpec: rows = concepts, cols = [label | pos imgs ... | gap | neg imgs ...]
# We model the gap as an extra narrow column.
col_widths = [LABEL_W] + [IMG_SIZE] * N_IMAGES + [GROUP_GAP] + [IMG_SIZE] * N_IMAGES
gs = gridspec.GridSpec(
    N_TOP_CONCEPTS + 1,          # +1 for header row
    len(col_widths),
    figure=fig,
    width_ratios=col_widths,
    wspace=PAD,
    hspace=PAD,
)

# ── header ───────────────────────────────────────────────────────
ax_header_pos = fig.add_subplot(gs[0, 1 : 1 + N_IMAGES])
ax_header_pos.axis("off")
ax_header_pos.set_title(pos_label, fontsize=13, fontweight="bold", pad=4)

ax_header_neg = fig.add_subplot(gs[0, 2 + N_IMAGES :])
ax_header_neg.axis("off")
ax_header_neg.set_title(neg_label, fontsize=13, fontweight="bold", pad=4)

# ── image rows ───────────────────────────────────────────────────
for row_i, (pos_concept, neg_concept) in enumerate(zip(sorted_pos_concepts, sorted_neg_concepts)):
    grid_row = row_i + 1   # row 0 is header

    # Concept label (left-most column, centred vertically)
    ax_label = fig.add_subplot(gs[grid_row, 0])
    ax_label.axis("off")
    pos_name = concept_names.get(int(pos_concept), f"c{int(pos_concept)}")
    neg_name = concept_names.get(int(neg_concept), f"c{int(neg_concept)}")
    ax_label.text(
        0.95, 0.7,
        f"[+] {pos_name}\n(w={weights[pos_concept]:+.2f})",
        ha="right", va="center", fontsize=7.5, transform=ax_label.transAxes,
        color="#1a6faf",
    )
    ax_label.text(
        0.95, 0.3,
        f"[−] {neg_name}\n(w={weights[neg_concept]:+.2f})",
        ha="right", va="center", fontsize=7.5, transform=ax_label.transAxes,
        color="#c0392b",
    )

    # Positive-class images
    for col_i, (img, img_idx) in enumerate(zip(*pos_images_per_concept[row_i])):
        ax = fig.add_subplot(gs[grid_row, 1 + col_i])
        ax.imshow(img)
        ax.set_title(str(img_idx), fontsize=5, pad=1)   # ← add this
        ax.axis("off")
        if col_i == 0:
            ax.set_ylabel(f"rank {row_i+1}", fontsize=7, labelpad=2)

    # Negative-class images
    for col_i, (img, img_idx) in enumerate(zip(*neg_images_per_concept[row_i])):
        ax = fig.add_subplot(gs[grid_row, 2 + N_IMAGES + col_i])
        ax.imshow(img)
        ax.set_title(str(img_idx), fontsize=5, pad=1)   # ← add this
        ax.axis("off")

fig.suptitle(
    f"Top {N_TOP_CONCEPTS} concepts × top {N_IMAGES} activating images  —  {ATTRIBUTE}",
    fontsize=14, fontweight="bold", y=1.01,
)

plt.savefig(f"concept_vis_{ATTRIBUTE}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → concept_vis_{ATTRIBUTE}.png")

Saved → concept_vis_Attractive.png


#### Sanity Checks

In [ ]:
# # Sanity check: Global top-activating images (no class restriction)
# # Border colour = ground-truth label
#
# N_IMAGES_GLOBAL = 8   # top images to show per concept globally
#
# img_files_arr = np.array(img_files)  # already defined in Cell 5, just making explicit
#
# def get_top_images_global(concept_idx, n=8):
#     """Top N images globally across the entire split, regardless of label."""
#     acts = sae_acts[:, concept_idx]
#     top_idx = torch.argsort(acts, descending=True)[:n]
#     return top_idx  # global split indices
#
# # Use the same top concepts from Cell 4
# # sorted_pos_concepts → concepts predicting positive class
# # sorted_neg_concepts → concepts predicting negative class
# all_concepts = list(sorted_pos_concepts) + list(sorted_neg_concepts)
# concept_labels_text = (
#         [f"[+] {concept_names.get(int(c), f'c{int(c)}')}\nw={weights[c]:+.2f}" for c in sorted_pos_concepts] +
#         [f"[−] {concept_names.get(int(c), f'c{int(c)}')}\nw={weights[c]:+.2f}" for c in sorted_neg_concepts]
# )
# concept_colors = ["#1a6faf"] * N_TOP_CONCEPTS + ["#c0392b"] * N_TOP_CONCEPTS
#
# # ── plot ──────────────────────────────────────────────────────────
# n_rows   = len(all_concepts)
# LABEL_W  = 2.0
# IMG_SIZE = 1.5
# PAD      = 0.05
#
# total_w = LABEL_W + N_IMAGES_GLOBAL * IMG_SIZE
# total_h = n_rows * IMG_SIZE
#
# fig2, axes = plt.subplots(
#     n_rows, N_IMAGES_GLOBAL + 1,   # +1 for label column
#     figsize=(total_w, total_h),
#     gridspec_kw={"width_ratios": [LABEL_W] + [IMG_SIZE] * N_IMAGES_GLOBAL,
#                  "wspace": PAD, "hspace": PAD}
# )
#
# for row_i, (concept, label_text, label_color) in enumerate(
#         zip(all_concepts, concept_labels_text, concept_colors)
# ):
#     top_idx = get_top_images_global(concept, n=N_IMAGES_GLOBAL)
#
#     # Concept label column
#     ax_label = axes[row_i, 0]
#     ax_label.axis("off")
#     ax_label.text(
#         0.95, 0.5, label_text,
#         ha="right", va="center", fontsize=7.5,
#         transform=ax_label.transAxes, color=label_color,
#     )
#
#     # Image columns
#     for col_i, img_idx in enumerate(top_idx):
#         ax = axes[row_i, col_i + 1]
#         img   = load_image(img_files_arr[img_idx])
#         true_label = labels[img_idx].item()
#
#         ax.imshow(img)
#         ax.set_xticks([]); ax.set_yticks([])
#
#         # Green border = truly positive class, red = truly negative class
#         border_color = "#2ecc71" if true_label == 1 else "#e74c3c"
#         for spine in ax.spines.values():
#             spine.set_visible(True)
#             spine.set_edgecolor(border_color)
#             spine.set_linewidth(4)
#
#         fname = img_files_arr[img_idx]
#         act_val = sae_acts[img_idx, concept].item()
#         ax.set_title(f"{fname}\nact={act_val:.1f}", fontsize=4, pad=1)
#
# fig2.suptitle(
#     f"Global top activating images — {ATTRIBUTE}\n"
#     f"green = truly {pos_label}   red = truly {neg_label}",
#     fontsize=12, fontweight="bold", y=1.01,
# )
#
# plt.savefig(f"concept_vis_{ATTRIBUTE}_global.png", dpi=150, bbox_inches="tight")
# plt.show()
# print(f"Saved → concept_vis_{ATTRIBUTE}_global.png")

In [ ]:
# # ══════════════════════════════════════════════════════════════════════════════
# # Cell: Visualise top-activating images per concept — AFTER unlearning
# #
# # What this does:
# #   1. Loads the unlearned probe weights  (binary_probe_finetuned_k<N>.pt)
# #   2. Re-ranks all concepts by |new weight| → new top-N concepts
# #   3. For each top concept: finds the images with highest activation for it
# #   4. Saves one PNG grid per concept  →  outputs/Mouth_Slightly_Open/vis_unlearned/
# #   5. Also saves updated top_concepts_unlearned.txt + all_concepts_ranked_unlearned.csv
# # ══════════════════════════════════════════════════════════════════════════════
#
# import os, csv
# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from PIL import Image
# from pathlib import Path
# from torchvision import transforms
#
# # ── 0. Paths — edit these to match your setup ────────────────────────────────
# ATTRIBUTE          = "Pale_Skin"
# OUTPUT_DIR         = f"./train_linear_probe_celeba/outputs/{ATTRIBUTE}"
# UNLEARNED_CKPT     = f"./unlearning/outputs/{ATTRIBUTE}/binary_probe_unlearned_k100.pt"
# CONCEPT_NAMES_CSV  = "./checkpoints/clip_RN50_concept_name.csv"   # idx,name,sim
# SAE_ACTIVATIONS    = "./data/activations_img/celeba/clip_RN50/out/train/sae_activations.pth"        # .pth tensor [N, 8192]
# CELEBA_IMG_DIR     = "./data/celeba/img_align_celeba"             # folder of .jpg files
# CELEBA_SPLIT_FILE  = "./data/celeba/list_eval_partition.txt"      # or Anno/list_eval_partition.txt
#
# VIS_DIR            = os.path.join(OUTPUT_DIR, "vis_unlearned")
# TOP_N_CONCEPTS     = 10     # how many top concepts to visualise
# IMGS_PER_CONCEPT   = 9      # images shown per concept (rows × cols below)
# GRID_ROWS, GRID_COLS = 3, 3
# assert GRID_ROWS * GRID_COLS == IMGS_PER_CONCEPT
#
# os.makedirs(VIS_DIR, exist_ok=True)
#
# # ── 1. Load concept names ─────────────────────────────────────────────────────
# concept_names = {}
# if os.path.exists(CONCEPT_NAMES_CSV):
#     with open(CONCEPT_NAMES_CSV, newline="") as f:
#         for row in csv.reader(f):
#             if len(row) >= 2:
#                 try:
#                     concept_names[int(row[0])] = row[1].strip()
#                 except ValueError:
#                     pass
# print(f"✓ Loaded {len(concept_names)} concept names")
#
# # ── 2. Load unlearned probe weights ──────────────────────────────────────────
# ckpt    = torch.load(UNLEARNED_CKPT, map_location="cpu")
# weights = ckpt["model_state"]["linear.weight"].squeeze()   # [n_concepts]
# n_concepts = weights.shape[0]
# print(f"✓ Unlearned weights loaded — shape: {weights.shape}")
#
# # ── 3. Re-rank concepts by |weight| and write updated CSVs ───────────────────
# sorted_indices = torch.argsort(weights.abs(), descending=True)
#
# # top_concepts_unlearned.txt
# top_txt_path = os.path.join(OUTPUT_DIR, "top_concepts_unlearned.txt")
# with open(top_txt_path, "w") as f:
#     for rank, idx in enumerate(sorted_indices[:TOP_N_CONCEPTS]):
#         w    = weights[idx].item()
#         name = concept_names.get(int(idx), f"concept_{int(idx)}")
#         line = f"  {rank+1:2d}. {name:<35} (idx={int(idx)}) | Weight: {w:+.6f}"
#         f.write(line + "\n")
#         print(line)
# print(f"\n✓ Saved → {top_txt_path}")
#
# # all_concepts_ranked_unlearned.csv
# ranked_csv_path = os.path.join(OUTPUT_DIR, "all_concepts_ranked_unlearned.csv")
# with open(ranked_csv_path, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["rank", "concept_idx", "name", "weight"])
#     for rank, idx in enumerate(sorted_indices):
#         idx_int = int(idx)
#         writer.writerow([rank + 1, idx_int,
#                          concept_names.get(idx_int, ""),
#                          f"{weights[idx].item():+.6f}"])
# print(f"✓ Saved → {ranked_csv_path}")
#
# # ── 4. Load SAE concept-strength activations ─────────────────────────────────
# print(f"\n→ Loading concept strengths from: {SAE_ACTIVATIONS}")
# all_concepts = torch.load(SAE_ACTIVATIONS, map_location="cpu")   # [N, n_concepts]
# if all_concepts.ndim == 3 and all_concepts.shape[1] == 1:
#     all_concepts = all_concepts.squeeze(1)
# print(f"✓ Concept strengths shape: {all_concepts.shape}")
# N = all_concepts.shape[0]
#
# # ── 5. Build ordered list of CelebA train image filenames ────────────────────
# # Reads list_eval_partition.csv (or .txt) to find partition==0 (train) images
# # and preserves the same order as the saved activations.
# img_files = []
# partition_path = Path(CELEBA_SPLIT_FILE)
# if partition_path.suffix == ".csv":
#     with open(partition_path, newline="") as f:
#         reader = csv.reader(f)
#         next(reader, None)   # skip header if present
#         for row in reader:
#             if len(row) >= 2 and row[1].strip() == "0":
#                 img_files.append(row[0].strip())
# else:   # plain txt: "000001.jpg 0"
#     with open(partition_path) as f:
#         for line in f:
#             parts = line.strip().split()
#             if len(parts) >= 2 and parts[1] == "0":
#                 img_files.append(parts[0])
#
# assert len(img_files) == N, (
#     f"Image list length ({len(img_files)}) ≠ activations length ({N}). "
#     "Check CELEBA_SPLIT_FILE and SAE_ACTIVATIONS are aligned to the same split/order.")
# print(f"✓ {len(img_files)} train images indexed")
#
# # ── 6. CLIP un-normalisation (same as vis scripts in this repo) ───────────────
# un_normalize = transforms.Normalize(
#     mean=(-0.48145466/0.26862954, -0.4578275/0.26130258, -0.40821073/0.27577711),
#     std =(1/0.26862954, 1/0.26130258, 1/0.27577711)
# )
# clip_preprocess = transforms.Compose([
#     transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
#                          (0.26862954, 0.26130258, 0.27577711)),
# ])
#
# def load_celeba_img(fname):
#     """Load, preprocess then un-normalise a CelebA image → numpy [H,W,3] in [0,1]."""
#     path = os.path.join(CELEBA_IMG_DIR, fname)
#     img  = Image.open(path).convert("RGB")
#     t    = clip_preprocess(img)           # [3,224,224] normalised
#     t    = un_normalize(t)                # back to [0,1] range
#     return t.permute(1, 2, 0).numpy().clip(0, 1)
#
# # ── 7. For each top concept: find top-activating images + save PNG ────────────
# top_concept_indices = sorted_indices[:TOP_N_CONCEPTS]
#
# for rank, concept_idx in enumerate(top_concept_indices):
#     c_idx  = int(concept_idx)
#     w_val  = weights[concept_idx].item()
#     c_name = concept_names.get(c_idx, f"concept_{c_idx}")
#
#     # top images for this concept
#     activations_for_concept = all_concepts[:, c_idx]          # [N]
#     _, top_img_idxs = torch.topk(activations_for_concept, k=IMGS_PER_CONCEPT)
#     top_img_idxs    = top_img_idxs.tolist()
#
#     # build grid
#     fig = plt.figure(figsize=(GRID_COLS * 2.5, GRID_ROWS * 2.5 + 0.8))
#     fig.suptitle(
#         f"Rank {rank+1}  |  \"{c_name}\"  (idx={c_idx})  |  weight={w_val:+.4f}",
#         fontsize=11, fontweight="bold", y=0.98
#     )
#     gs  = gridspec.GridSpec(GRID_ROWS, GRID_COLS, figure=fig,
#                             hspace=0.05, wspace=0.05)
#
#     for pos, img_idx in enumerate(top_img_idxs):
#         ax  = fig.add_subplot(gs[pos // GRID_COLS, pos % GRID_COLS])
#         img = load_celeba_img(img_files[img_idx])
#         ax.imshow(img)
#         act_val = activations_for_concept[img_idx].item()
#         ax.set_title(f"act={act_val:.2f}", fontsize=7, pad=2)
#         ax.axis("off")
#
#     png_path = os.path.join(VIS_DIR, f"rank{rank+1:02d}_concept{c_idx}_{c_name.replace(' ','_')}.png")
#     fig.savefig(png_path, dpi=120, bbox_inches="tight")
#     plt.close(fig)
#     print(f"  [{rank+1:2d}/{TOP_N_CONCEPTS}] Saved → {png_path}")
#
# print(f"\n✓ All {TOP_N_CONCEPTS} concept PNGs saved to: {VIS_DIR}")

In [ ]:
# # Debug cell — verify label sanity
# import os
# from pathlib import Path
#
# celeba_root = "./data/celeba"
# attr_file = Path(celeba_root) / "list_attr_celeba.txt"
# with open(attr_file) as f:
#     lines = f.readlines()
#
# attr_names = lines[1].strip().split()
# attr_idx   = attr_names.index("Blond_Hair")
# print(f"Blond_Hair is column index {attr_idx}")
#
# # Check first 10 images — print filename + raw attribute value + label
# partition_file = Path(celeba_root) / "list_eval_partition.txt"
# with open(partition_file) as f:
#     part_lines = f.readlines()
#
# print(f"\n{'filename':<20} {'raw_val':>8} {'label':>6}")
# print("-" * 38)
# count = 0
# for img_idx, part_line in enumerate(part_lines):
#     parts = part_line.strip().split()
#     if int(parts[-1]) == 0:   # train split
#         fname   = parts[0]
#         raw_val = int(lines[2 + img_idx].strip().split()[attr_idx])
#         label   = 1 if raw_val == 1 else 0
#         print(f"{fname:<20} {raw_val:>8} {label:>6}")
#         count += 1
#         if count >= 10:
#             break

### check attributes for specific data

In [42]:
# def get_attributes_for_index(split_idx, celeba_root, split="train"):
#     """
#     Given a split-local index (as used in sae_activations), return
#     all 40 CelebA attributes for that image as a dict {attr_name: 0/1}.
#     """
#     # Step 1: map split index → global image index
#     split_map = {"train": 0, "val": 1, "test": 2}
#     target = split_map[split]
#     partition_file = Path(celeba_root) / "list_eval_partition.txt"
#     with open(partition_file) as f:
#         lines = f.readlines()
#
#     global_indices = [i for i, l in enumerate(lines) if int(l.strip().split()[-1]) == target]
#     global_idx = global_indices[split_idx]
#
#     # Step 2: read attribute line for that global image
#     attr_file = Path(celeba_root) / "list_attr_celeba.txt"
#     with open(attr_file) as f:
#         attr_lines = f.readlines()
#
#     attr_names = attr_lines[1].strip().split()
#     attr_vals = attr_lines[2 + global_idx].strip().split()[1:]  # ← skip filename   # line 0=count, 1=names, 2+=data
#
#     return {name: (1 if int(val) == 1 else 0) for name, val in zip(attr_names, attr_vals)}

In [49]:
# # look up all attributes for split index 42
# attrs = get_attributes_for_index(6890, CELEBA_ROOT, split=SPLIT)
#
# # print only the active ones (value == 1)
# active = [k for k, v in attrs.items() if v == 1]
# print("Active attributes:", active)
#
# # or print all
# for name, val in attrs.items():
#     print(f"  {name:<25} {'✓' if val else '✗'}")

Active attributes: ['Attractive', 'Brown_Hair', 'Bushy_Eyebrows', 'Heavy_Makeup', 'High_Cheekbones', 'No_Beard', 'Rosy_Cheeks', 'Smiling', 'Wavy_Hair', 'Wearing_Lipstick', 'Young']
  5_o_Clock_Shadow          ✗
  Arched_Eyebrows           ✗
  Attractive                ✓
  Bags_Under_Eyes           ✗
  Bald                      ✗
  Bangs                     ✗
  Big_Lips                  ✗
  Big_Nose                  ✗
  Black_Hair                ✗
  Blond_Hair                ✗
  Blurry                    ✗
  Brown_Hair                ✓
  Bushy_Eyebrows            ✓
  Chubby                    ✗
  Double_Chin               ✗
  Eyeglasses                ✗
  Goatee                    ✗
  Gray_Hair                 ✗
  Heavy_Makeup              ✓
  High_Cheekbones           ✓
  Male                      ✗
  Mouth_Slightly_Open       ✗
  Mustache                  ✗
  Narrow_Eyes               ✗
  No_Beard                  ✓
  Oval_Face                 ✗
  Pale_Skin                 ✗
  Pointy_

In [ ]:
# # Quick label sanity check — run this in the notebook
# with open("./data/celeba/list_attr_celeba.txt") as f:
#     lines = f.readlines()
#
# attr_names = lines[1].strip().split()
# attr_idx = attr_names.index("Blond_Hair")
# print(f"Blond_Hair is attribute index {attr_idx}")
#
# # Check first image
# first_line = lines[1].strip().split()
# print(f"Filename: {first_line[0]}")
# print(f"attr_line[attr_idx]     = {first_line[attr_idx]}    ← what the bug reads")
# print(f"attr_line[attr_idx + 1] = {first_line[attr_idx+1]}  ← correct value")

In [ ]:
# with open("./data/celeba/list_attr_celeba.txt") as f:
#     lines = f.readlines()
#
# attr_names = lines[1].strip().split()
# attr_idx = attr_names.index("Blond_Hair")
# print(f"Blond_Hair attr_idx = {attr_idx}")
#
# # Check a known-blonde image — 000001.jpg is NOT blonde, try 000002.jpg
# for i in range(2, 12):
#     row = lines[i].strip().split()
#     filename = row[0]
#     val_at_idx        = row[attr_idx]      # what data_loader reads
#     val_at_idx_plus1  = row[attr_idx + 1]  # shifted
#     print(f"{filename}  | attr_line[{attr_idx}]={val_at_idx:3s}  | attr_line[{attr_idx+1}]={val_at_idx_plus1:3s}")

In [ ]:
# # Run this in the notebook
#
# import torch
# from pathlib import Path
#
# CELEBA_ROOT   = "./data/celeba"
# SAE_ACT_TRAIN = "./data/activations_img/celeba/clip_RN50/out/train/sae_activations.pth"
#
# # ── 1. Order that extract_celeba_clip_sae.py saved activations ───
# # (reads list_eval_partition.txt, keeps only partition==0)
# split_file = Path(CELEBA_ROOT) / "list_eval_partition.txt"
# activation_order = []   # filenames in the order activations were saved
# with open(split_file) as f:
#     for line in f:
#         parts = line.strip().split()
#         if len(parts) >= 2 and int(parts[1]) == 0:   # 0 = train
#             activation_order.append(parts[0])
#
# # ── 2. Order that data_loader.py assigns labels ──────────────────
# # (reads list_eval_partition.txt the same way → should match)
# label_order = activation_order   # same loop, so same order
#
# # ── 3. Load attr file and check the first 10 blonde images ───────
# with open(Path(CELEBA_ROOT) / "list_attr_celeba.txt") as f:
#     attr_lines = f.readlines()
#
# attr_names = attr_lines[1].strip().split()
# blond_idx  = attr_names.index("Blond_Hair")
#
# # Build a filename→label dict from the attr file
# attr_dict = {}
# for line in attr_lines[2:]:
#     parts = line.strip().split()
#     fname = parts[0]
#     attr_dict[fname] = int(parts[blond_idx])
#
# # ── 4. Compare first 20 entries ──────────────────────────────────
# print(f"{'Rank':>5} | {'Filename':<15} | {'Label from attr_dict':>20} | {'In activation_order'}")
# print("-" * 70)
# for i, fname in enumerate(activation_order[:20]):
#     label = attr_dict.get(fname, "???")
#     print(f"{i:5d} | {fname:<15} | {str(label):>20} | ✓")
#
# # ── 5. Check if the attr file has a HEADER filename row ──────────
# print(f"\nFirst 3 raw lines of list_attr_celeba.txt:")
# for line in attr_lines[:3]:
#     print(repr(line[:80]))

In [ ]:
# import torch
#
# ckpt = torch.load("train_linear_probe_celeba/outputs/Blond_Hair/binary_probe.pt", map_location="cpu")
# weights = ckpt["weights"].squeeze()
#
# # Load activations and labels
# sae_acts = torch.load("./data/activations_img/celeba/clip_RN50/out/train/sae_activations.pth", map_location="cpu")
# if sae_acts.ndim == 3:
#     sae_acts = sae_acts.squeeze(1)
#
# from train_linear_probe_celeba.data_loader import load_celeba_labels
# labels = load_celeba_labels("./data/celeba", split="train", attribute="Blond_Hair")
#
# # Compute logits manually
# logits = sae_acts @ weights   # [N]
#
# # Check: do positive logits correspond to blonde (label=1)?
# preds = (logits > 0).long()
# acc       = (preds == labels).float().mean().item()
# acc_flip  = (preds != labels).float().mean().item()   # accuracy if we flip
#
# n_blonde     = (labels == 1).sum().item()
# n_pred_pos   = (preds == 1).sum().item()
#
# print(f"Accuracy as-is:    {acc:.4f}")
# print(f"Accuracy if flip:  {acc_flip:.4f}")
# print(f"Actual blonde:     {n_blonde}")
# print(f"Predicted positive:{n_pred_pos}")
# print(f"\nMean logit for blonde    (label=1): {logits[labels==1].mean():.4f}")
# print(f"Mean logit for not-blonde(label=0): {logits[labels==0].mean():.4f}")

In [ ]:
# # In notebook — verify pos_idx really are blonde
# sample = pos_idx[:5]
# for i in sample:
#     fname = img_files_arr[i]
#     label = labels[i].item()
#     print(f"  {fname}  label={label}  (1=blonde)")

In [ ]:
# with open(Path(CELEBA_ROOT) / "list_attr_celeba.txt") as f:
#     lines = f.readlines()
#
# attr_names = lines[1].strip().split()
# print(f"Number of attr_names: {len(attr_names)}")  # should be 40
# print(f"First token of data line: {lines[2].strip().split()[0]}")  # should be filename
# print(f"attr_names[0]: {attr_names[0]}")  # should be 5_o_Clock_Shadow
#
# # Check 000007.jpg which is known blonde
# row = lines[8].strip().split()   # 000007.jpg is line index 8 (lines[2+6])
# print(f"\nRow for 000007.jpg:")
# print(f"  row[0]  = {row[0]}   ← filename")
# print(f"  row[9]  = {row[9]}   ← attr_idx=9, no +1")
# print(f"  row[10] = {row[10]}  ← attr_idx+1")
# print(f"  Expected Blond_Hair=1 at which position?")

In [ ]:
# from pathlib import Path
#
# CELEBA_ROOT = "./data/celeba"
# split_file = Path(CELEBA_ROOT) / "list_eval_partition.txt"
#
# with open(split_file) as f:
#     lines = [l.strip() for l in f if l.strip()]
#
# # Method A: load_celeba_filenames
# method_a = []
# for l in lines:
#     parts = l.split()
#     if int(parts[1]) == 0:
#         method_a.append(parts[0])
#
# # Method B: load_celeba_labels (just partition list, then enumerate)
# partitions = [int(l.split()[-1]) for l in lines]
# method_b_files = []
# for img_idx, part in enumerate(partitions):
#     if part == 0:
#         method_b_files.append(lines[img_idx].split()[0])
#
# print(f"Method A count: {len(method_a)}")
# print(f"Method B count: {len(method_b_files)}")
# print(f"First match: {method_a[0] == method_b_files[0]}")
# print(f"Last match:  {method_a[-1] == method_b_files[-1]}")
# print(f"All match:   {method_a == method_b_files}")
#
# # Also check sae_acts count
# import torch
# sae_acts = torch.load("./data/activations_img/celeba/clip_RN50/out/train/sae_activations.pth", map_location="cpu")
# if sae_acts.ndim == 3:
#     sae_acts = sae_acts.squeeze(1)
# print(f"\nSAE activations count: {sae_acts.shape[0]}")
# print(f"Method A count:        {len(method_a)}")
# print(f"Match: {sae_acts.shape[0] == len(method_a)}")

In [ ]:
# from pathlib import Path
# from PIL import Image
# import matplotlib.pyplot as plt
#
# # Pick 4 known-blonde (label=1) and 4 known-not-blonde (label=0)
# pos_sample_idx = pos_idx[:4].tolist()
# neg_sample_idx = neg_idx[:4].tolist()
#
# fig, axes = plt.subplots(2, 4, figsize=(10, 5))
#
# for col, i in enumerate(pos_sample_idx):
#     fname = img_files_arr[i]
#     img = Image.open(Path(CELEBA_ROOT) / "img_align_celeba" / fname).convert("RGB")
#     axes[0, col].imshow(img)
#     axes[0, col].set_title(f"{fname}\nlabel=1 (blonde)", fontsize=7)
#     axes[0, col].axis("off")
#
# for col, i in enumerate(neg_sample_idx):
#     fname = img_files_arr[i]
#     img = Image.open(Path(CELEBA_ROOT) / "img_align_celeba" / fname).convert("RGB")
#     axes[1, col].imshow(img)
#     axes[1, col].set_title(f"{fname}\nlabel=0 (not blonde)", fontsize=7)
#     axes[1, col].axis("off")
#
# axes[0, 0].set_ylabel("label=1\n(blonde)", fontsize=9)
# axes[1, 0].set_ylabel("label=0\n(not blonde)", fontsize=9)
#
# plt.tight_layout()
# plt.savefig("label_sanity.png", dpi=150, bbox_inches="tight")
# plt.show()

In [ ]:
# # What is split-local index 113960 actually pointing to?
# print(f"img_files_arr[113960] = {img_files_arr[113960]}")
#
# # What does get_attributes_for_index(113960) look up?
# # Show us its implementation
# import inspect
# # paste output of:
# print(inspect.getsource(get_attributes_for_index))